# PROYECTO DE TESIS — UNIVERSIDAD DE LIMA
## Modelo de Evaluación de Riesgo Crediticio para Microfinanzas mediante XGBoost e Inteligencia Artificial Explicable (SHAP)
**Autor:** Joaquín Ignacio Bernales Demichelli  
**Metodología:** CRISP-DM (*Cross-Industry Standard Process for Data Mining*)  
**Alineación:** ODS 8 (Trabajo Decente y Crecimiento Económico) y ODS 10 (Reducción de Desigualdades)

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.feature_engineering import add_engineered_features
from src.data_preprocessing import CreditDataPreprocessor, stratified_split_dataset
from src.model_training import CreditRiskXGBoostWrapper
from src.explainability import CreditRiskExplainer
from src.model_evaluation import compute_classification_metrics

print("Entorno cargado exitosamente.")

### 1. Carga del Conjunto de Datos e Inspección Exploratoria (EDA)

In [ ]:
df_raw = pd.read_csv(config.RAW_DATASET_PATH)
print(f"Dimensiones: {df_raw.shape}")
print(f"Tasa de mora (default rate): {df_raw[config.TARGET_COLUMN].mean()*100:.2f}%")
df_raw.head()

### 2. Ingeniería de Ratios Financieros y Variables Alternativas Thin-File

In [ ]:
df_feat = add_engineered_features(df_raw)
print("Variables derivadas calculadas:")
df_feat[config.NUMERICAL_DERIVED_FEATURES].describe().T

### 3. Partición Estratificada y Preprocesamiento

In [ ]:
df_train, df_val, df_test = stratified_split_dataset(df_feat)
preprocessor = CreditDataPreprocessor()
X_train = preprocessor.fit_transform(df_train)
y_train = df_train[config.TARGET_COLUMN].values

X_test = preprocessor.transform(df_test)
y_test = df_test[config.TARGET_COLUMN].values
print(f"Dimensiones X_train: {X_train.shape} | X_test: {X_test.shape}")

### 4. Entrenamiento del Modelo XGBoost con Cost-Sensitive Learning

In [ ]:
model = CreditRiskXGBoostWrapper()
model.fit(X_train, y_train, feature_names=preprocessor.get_feature_names_out())

y_prob = model.predict_proba(X_test)[:, 1]
metrics = compute_classification_metrics(y_test, y_prob, threshold=0.50)
print("Métricas en Test Set:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

### 5. Explicabilidad Algorítmica con SHAP (TreeSHAP)

In [ ]:
explainer = CreditRiskExplainer(model, preprocessor)
df_global_imp = explainer.get_global_feature_importance(df_test, top_n=10)
print("Top 10 Variables Globales más importantes:")
df_global_imp